# Build the Vietnamese legal vector index on a Colab GPU

In VS Code choose **Select Kernel → Colab**, sign in, and connect to a **GPU** runtime (a T4 is sufficient). Then run the cells in order.

This notebook clones the original repository and overlays a verified snapshot of our current local changes. It embeds no `.env`, API key, model weights, or dataset. Groq is not needed for indexing.

The result is `colab_vector_index.zip`, containing the complete portable Qdrant index and a build manifest. Download it through the Colab file browser or the final download cell. On your local machine, stop the chatbot and run `python import_vector_index.py <downloaded zip path>`. The existing partial index is backed up. The separately built local BM25 index is retained.


In [1]:
from pathlib import Path
import base64, hashlib, io, os, subprocess, zipfile
SOURCE_SHA256 = '9daa0dee906a9a7c64279e803b46967db3bb8e4d80d0e0fe87bdd2217f22ccd9'
SOURCE_BASE64 = '...'
source = base64.b64decode(SOURCE_BASE64)
assert hashlib.sha256(source).hexdigest() == SOURCE_SHA256
REPO = Path("/content/vietnamese-legal-chatbot")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/fisherman611/vietnamese-legal-chatbot.git", str(REPO)], check=True)
    subprocess.run(["git", "checkout", 'eea8848c6a8f6506cd88b2bef6ce2b76e0d5f92e'], cwd=REPO, check=True)
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    archive.extractall(REPO)
os.chdir(REPO)
print("Current source restored:", SOURCE_SHA256)

Current source restored: 9daa0dee906a9a7c64279e803b46967db3bb8e4d80d0e0fe87bdd2217f22ccd9


## Install dependencies and download the dataset

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "download_dataset.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'download_dataset.py'], returncode=0)

## Verify the GPU and build all 61,068 article vectors

In [3]:
import os, time, torch
os.environ["QDRANT_PATH"] = "index/qdrant"
os.environ.pop("QDRANT_URL", None)
os.environ.pop("QDRANT_API_KEY", None)
assert torch.cuda.is_available(), "Connect a GPU Colab runtime before indexing."
print("GPU:", torch.cuda.get_device_name(0))
from config import Config
from main.vector_store import QdrantVectorStore
from utils.data_loader import LegalDataLoader
Config.QDRANT_PATH = "index/qdrant"
Config.QDRANT_URL = Config.QDRANT_API_KEY = None
Config.EMBEDDING_BATCH_SIZE = 64
documents = LegalDataLoader().prepare_documents_for_indexing()
store = QdrantVectorStore()
assert store.embedding_model.device.type == "cuda", "Embedding model must use the GPU."
dimensions = store.embedding_model.get_embedding_dimension()
store.create_collection(vector_size=dimensions)
started = time.time()
store.add_documents(documents)
info = store.get_collection_info()
assert info["points_count"] == len(documents), "Index is incomplete. Rerun this cell to resume."
print("Indexed", len(documents), "articles in", round(time.time() - started, 1), "seconds")

GPU: Tesla T4
Loaded 3271 legal documents


  0%|          | 0/3271 [00:00<?, ?it/s]

Prepared 61068 documents for indexing
QDrant client initialized successfully


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

... (11 outputs left out)

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Embedding model bkai-foundation-models/vietnamese-bi-encoder loaded successfully
Collection exists: False
Collection bkai_biencoder_vietnamese_legal_corpus does not exist - creating new collection
Successfully created collection: bkai_biencoder_vietnamese_legal_corpus


Indexing articles:  33%|███▎      | 156/478 [07:17<15:08,  2.82s/it]/content/vietnamese-legal-chatbot/main/vector_store.py:164: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20096 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self.client.upsert(collection_name=self.collection_name, points=points, wait=True)
Indexing articles: 100%|██████████| 478/478 [22:32<00:00,  2.83s/it]

Successfully added 61068 new documents to vector store
Collection info: {'name': 'bkai_biencoder_vietnamese_legal_corpus', 'vectors_count': 61068, 'indexed_vectors_count': 0, 'points_count': 61068}
Indexed 61068 articles in 1352.5 seconds


## Check dense retrieval and the multilingual BGE reranker

In [4]:
from main.reranker import DocumentReranker
query = "Người lao động có quyền nhận lương đúng hạn không?"
candidates = store.search_similar_documents(query, top_k=20)
assert candidates, "Dense retrieval returned no documents."
reranker = DocumentReranker()
assert reranker.model is not None, "BGE reranker failed to load."
ranked = reranker.rerank_with_fusion(query, candidates, alpha=Config.RERANKER_FUSION_ALPHA, top_k=5)
assert ranked and all("reranker_score" in doc for doc in ranked)
for doc in ranked:
    print(doc["id"], round(doc["score"], 4), doc["title"])

Vector DB found 20 similar documents
Loading reranker model: BAAI/bge-reranker-v2-m3


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker model loaded successfully
Reranked 20 documents
Top reranker scores: [0.2241981327533722, 0.17924495041370392, 0.14942070841789246]
Score fusion reranking completed for 5 documents
27/2014/nđ-cp_15 0.9521 Điều 15. Tiền lương, hình thức trả lương, thời hạn trả lương
19/2014/tt-blđtbxh_13 0.6456 Điều 13. Hình thức trả lương, thời hạn trả lương
115/2015/nđ-cp_2 0.5318 Điều 2. Đối tượng áp dụng
84/2015/qh13_6 0.2621 Điều 6. Quyền và nghĩa vụ về an toàn, vệ sinh lao động của người lao động
20/2020/nđ-cp_10 0.2485 Điều 10. Trả lương, tiền thưởng


## Export the index and its build manifest

In [5]:
import importlib.metadata, json, zipfile
dataset_manifest = json.loads(Path("data/dataset_manifest.json").read_text())
manifest = {
    "embedding_model": Config.EMBEDDING_MODEL,
    "embedding_dimensions": dimensions,
    "collection": Config.COLLECTION_NAME,
    "article_count": len(documents),
    "dataset_archive_sha256": dataset_manifest["archive_sha256"],
    "source_sha256": SOURCE_SHA256,
    "reranker_model": Config.RERANKER_MODEL,
    "qdrant_client_version": importlib.metadata.version("qdrant-client"),
    "sentence_transformers_version": importlib.metadata.version("sentence-transformers"),
    "smoke_test_article_ids": [doc["id"] for doc in ranked],
}
store.client.close()  # Flush SQLite and release the local-store lock before exporting.
manifest_file = Path("index/build_manifest.json")
manifest_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
EXPORT = REPO / "colab_vector_index.zip"
with zipfile.ZipFile(EXPORT, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
    archive.write(manifest_file, "build_manifest.json")
    for file in Path(Config.QDRANT_PATH).rglob("*"):
        if file.is_file() and file.name != ".lock":
            archive.write(file, "qdrant/" + file.relative_to(Config.QDRANT_PATH).as_posix())
print("Download:", EXPORT, "Size MB:", round(EXPORT.stat().st_size / 1024**2, 1))
print(json.dumps(manifest, indent=2))

Download: /content/vietnamese-legal-chatbot/colab_vector_index.zip Size MB: 260.1
{
  "embedding_model": "bkai-foundation-models/vietnamese-bi-encoder",
  "embedding_dimensions": 768,
  "collection": "bkai_biencoder_vietnamese_legal_corpus",
  "article_count": 61068,
  "dataset_archive_sha256": "13e0d5cebfb0c3281a0f603ca6791547e78fe76a1aa86d0e89b39ae2f55a3913",
  "source_sha256": "9daa0dee906a9a7c64279e803b46967db3bb8e4d80d0e0fe87bdd2217f22ccd9",
  "reranker_model": "BAAI/bge-reranker-v2-m3",
  "qdrant_client_version": "1.19.1",
  "sentence_transformers_version": "5.7.0",
  "smoke_test_article_ids": [
    "27/2014/n\u0111-cp_15",
    "19/2014/tt-bl\u0111tbxh_13",
    "115/2015/n\u0111-cp_2",
    "84/2015/qh13_6",
    "20/2020/n\u0111-cp_10"
  ]
}


## Download to your local machine
If the VS Code output cannot trigger a download, use the Colab runtime file browser to download `/content/vietnamese-legal-chatbot/colab_vector_index.zip`.

In [6]:
from google.colab import files
files.download(str(EXPORT))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>